In [1]:
import openml

import pandas as pd
import numpy as np

In [2]:
def preprocesar_datos(df, null_threshold=0.75):
    df = df.copy()

    # 1. SUSTITUIR COLUMNAS CON MUCHOS NaN POR VARIABLES BINARIAS (nulo/no_nulo)
    null_percentages = df.isna().sum() / len(df)
    cols_to_drop = null_percentages[null_percentages > null_threshold].index.tolist()

    if cols_to_drop:
        new_cols = {f'{col}_is_null': df[col].isna().astype(int) for col in cols_to_drop}
        df_indicators = pd.DataFrame(new_cols)
        df = pd.concat([df, df_indicators], axis=1)
        df = df.drop(cols_to_drop, axis=1)

    # 2. IDENTIFICACIÓN DE VARIABLES CATEGÓRICAS Y NUMÉRICAS
    # Variables categóricas
    categorical_cols = []
    numeric_disguised_as_categorical = []

    for col in df.select_dtypes(exclude=[np.number]).columns.tolist():
        try:
            non_null_values = df[col].dropna()
            pd.to_numeric(non_null_values)
            numeric_disguised_as_categorical.append(col)
        except (ValueError, TypeError):
            categorical_cols.append(col)

    # Variables numéricas (sin contar las columnas _is_null)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if not c.endswith('_is_null')]
    numeric_cols = numeric_cols + numeric_disguised_as_categorical

    for col in numeric_disguised_as_categorical:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3. GESTIÓN DE LOS VALORES FALTANTES
    # Imputación de la mediana en variables numéricas
    for col in numeric_cols:
        if df[col].isna().any():
            n = df[col].isna().sum()
            mediana = df[col].median()
            df[col] = df[col].fillna(mediana)

    # Imputación de la moda en variables categóricas
    for col in categorical_cols:
        if df[col].isna().any():
            n = df[col].isna().sum()
            moda = df[col].mode()[0] if len(df[col].mode()) > 0 else None
            df[col] = df[col].fillna(moda)

    # 4. CODIFICACIÓN DE CATEGÓRICAS
    if categorical_cols:
        for col in categorical_cols:
            df[col] = df[col].astype('category').cat.codes.astype('category')

    return df

In [3]:
suite = openml.study.get_suite(271)
tasks = suite.tasks # 71 tasks en total

for task_id in tasks:
    task = openml.tasks.get_task(task_id)

    if len(task.class_labels) == 2: # Hay 41 datasets de clasificación binaria
        dataset = task.get_dataset()
        X, y, _, _ = dataset.get_data(target=task.target_name)
        print("Task:", task_id, "- Forma de X:", X.shape)

Task: 3945 - Forma de X: (50000, 230)
Task: 146818 - Forma de X: (690, 14)
Task: 146820 - Forma de X: (4839, 5)
Task: 167120 - Forma de X: (96320, 21)
Task: 168350 - Forma de X: (5404, 5)
Task: 168757 - Forma de X: (1000, 20)
Task: 168868 - Forma de X: (76000, 170)
Task: 168911 - Forma de X: (2984, 144)
Task: 189354 - Forma de X: (539383, 7)
Task: 189356 - Forma de X: (425240, 78)
Task: 189922 - Forma de X: (3153, 970)
Task: 190137 - Forma de X: (2534, 72)
Task: 190392 - Forma de X: (3140, 259)
Task: 190410 - Forma de X: (5832, 308)
Task: 190411 - Forma de X: (4147, 48)
Task: 190412 - Forma de X: (100, 10000)
Task: 359955 - Forma de X: (748, 4)
Task: 359956 - Forma de X: (1055, 41)
Task: 359958 - Forma de X: (1458, 37)
Task: 359962 - Forma de X: (2109, 21)
Task: 359965 - Forma de X: (3196, 36)
Task: 359966 - Forma de X: (3279, 1558)
Task: 359967 - Forma de X: (3751, 1776)
Task: 359968 - Forma de X: (5000, 20)
Task: 359971 - Forma de X: (11055, 30)
Task: 359972 - Forma de X: (5124, 20)


In [4]:
filtered_tasks = []

for task_id in tasks:
    task = openml.tasks.get_task(task_id)

    # Filtro de clasificación binaria 
    if len(task.class_labels) != 2: 
        continue

    dataset = task.get_dataset()
    q = dataset.qualities
    n, m = int(q['NumberOfInstances']), int(q['NumberOfFeatures'])

    # Filtro de tamaño:
    # 1. Filas
    if n < 1000 or n >= 50000:
        continue
    
    # 2. Columnas
    if m > 350:
        continue

    filtered_tasks.append(task.task_id)

print(len(filtered_tasks))
print(filtered_tasks)

21
[146820, 168350, 168757, 168911, 190137, 190392, 190410, 190411, 359956, 359958, 359962, 359965, 359968, 359971, 359972, 359975, 359979, 359980, 359982, 359983, 359992]


In [5]:
filtered_tasks_2 = [x for x in filtered_tasks if x != 359992]

In [6]:
for task_id in filtered_tasks_2:
    task = openml.tasks.get_task(task_id)
    dataset = task.get_dataset()
    X, y, _, _ = dataset.get_data(target=task.target_name)
    print("Task:", task_id, "- Forma de X:", X.shape)

Task: 146820 - Forma de X: (4839, 5)
Task: 168350 - Forma de X: (5404, 5)
Task: 168757 - Forma de X: (1000, 20)
Task: 168911 - Forma de X: (2984, 144)
Task: 190137 - Forma de X: (2534, 72)
Task: 190392 - Forma de X: (3140, 259)
Task: 190410 - Forma de X: (5832, 308)
Task: 190411 - Forma de X: (4147, 48)
Task: 359956 - Forma de X: (1055, 41)
Task: 359958 - Forma de X: (1458, 37)
Task: 359962 - Forma de X: (2109, 21)
Task: 359965 - Forma de X: (3196, 36)
Task: 359968 - Forma de X: (5000, 20)
Task: 359971 - Forma de X: (11055, 30)
Task: 359972 - Forma de X: (5124, 20)
Task: 359975 - Forma de X: (5100, 36)
Task: 359979 - Forma de X: (32769, 9)
Task: 359980 - Forma de X: (34465, 118)
Task: 359982 - Forma de X: (45211, 16)
Task: 359983 - Forma de X: (48842, 14)


In [7]:
# 1. Creamos una lista vacía para recolectar los datos
results = []

for task_id in filtered_tasks_2:
    task = openml.tasks.get_task(task_id)
    dataset = task.get_dataset()
    
    # Extraemos el nombre del dataset
    dataset_name = dataset.name
    
    # Extraemos X y obtenemos su forma
    X, y, _, _ = dataset.get_data(target=task.target_name)
    
    # 2. Guardamos la información en un diccionario (añadiendo el Nombre)
    results.append({
        "Task ID": task_id,
        "Nombre": dataset_name,
        "Tamaño": X.shape
    })

# 3. Convertimos la lista completa en un DataFrame
df_tasks = pd.DataFrame(results)

In [8]:
display(df_tasks[:10]), display(df_tasks[10:])

,Task ID,Nombre,Tamaño
0,146820,wilt,"(4839, 5)"
1,168350,phoneme,"(5404, 5)"
2,168757,credit-g,"(1000, 20)"
3,168911,jasmine,"(2984, 144)"
4,190137,ozone-level-8hr,"(2534, 72)"
5,190392,madeline,"(3140, 259)"
6,190410,philippine,"(5832, 308)"
7,190411,ada,"(4147, 48)"
8,359956,qsar-biodeg,"(1055, 41)"
9,359958,pc4,"(1458, 37)"


,Task ID,Nombre,Tamaño
10,359962,kc1,"(2109, 21)"
11,359965,kr-vs-kp,"(3196, 36)"
12,359968,churn,"(5000, 20)"
13,359971,PhishingWebsites,"(11055, 30)"
14,359972,sylvine,"(5124, 20)"
15,359975,Satellite,"(5100, 36)"
16,359979,Amazon_employee_access,"(32769, 9)"
17,359980,nomao,"(34465, 118)"
18,359982,bank-marketing,"(45211, 16)"
19,359983,adult,"(48842, 14)"


(None, None)

Nos quedamos con 20 datasets.

Veamos cúantos folds usa cada task en validación cruzada.

In [9]:
for task_id in filtered_tasks_2:
    task = openml.tasks.get_task(task_id)
    # Esta función devuelve (n_repeats, n_folds, n_samples)
    n_repeats, n_folds, n_samples = task.get_split_dimensions()
    print(f"{task_id} | {n_repeats} | {n_folds} | {n_samples}")

146820 | 1 | 10 | 1
168350 | 1 | 10 | 1
168757 | 1 | 10 | 1
168911 | 1 | 10 | 1
190137 | 1 | 10 | 1
190392 | 1 | 10 | 1
190410 | 1 | 10 | 1
190411 | 1 | 10 | 1
359956 | 1 | 10 | 1
359958 | 1 | 10 | 1
359962 | 1 | 10 | 1
359965 | 1 | 10 | 1
359968 | 1 | 10 | 1
359971 | 1 | 10 | 1
359972 | 1 | 10 | 1
359975 | 1 | 10 | 1
359979 | 1 | 10 | 1
359980 | 1 | 10 | 1
359982 | 1 | 10 | 1
359983 | 1 | 10 | 1
